In [0]:
%pip install pydeequ==1.3.0
%restart_python

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
#Code is compatible with 3.1,3.2,3.3 version
%env SPARK_VERSION=3.1
#import os
#os.environ['SPARK_VERSION'] = '3.3'

env: SPARK_VERSION=3.1


In [0]:
import os

# Must be set BEFORE importing pydeequ. Match the Deequ jar on the cluster.
#os.environ["SPARK_VERSION"] = spark.version[:3]   # e.g. "3.4" or "3.5"
#print("Spark version:", spark.version, "| SPARK_VERSION env:", os.environ["SPARK_VERSION"])

import random
from datetime import date, datetime, timedelta

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T

from pydeequ.checks import Check, CheckLevel
from pydeequ.verification import VerificationSuite, VerificationResult
from pydeequ.repository import FileSystemMetricsRepository, ResultKey
from pydeequ.analyzers import (
    AnalysisRunner, AnalyzerContext, Size, Completeness, Uniqueness,
    ApproxCountDistinct, Mean, StandardDeviation, Compliance,
)
from pydeequ.anomaly_detection import RelativeRateOfChangeStrategy
from pydeequ.suggestions import ConstraintSuggestionRunner, DEFAULT

# Sanity check that the Deequ jar is on the classpath
spark._jvm.com.amazon.deequ.checks.CheckLevel.Error()
print("Deequ JVM classes found.")


Deequ JVM classes found.


In [0]:

# N_ORDERS = 1000
# N_CUSTOMERS = 300
# SEED = 42

# # Where Deequ stores historical metrics (any Hadoop-compatible path).
# # Unity Catalog volume example: "/Volumes/main/default/dq/metrics_history.json"
# METRICS_PATH = "dbfs:/tmp/pydeequ_demo/metrics_history.json"

# # Optionally persist datasets + quarantine as Delta tables
# WRITE_TABLES = True
# TARGET_SCHEMA = "asper_production_asper_us_dp_dev_catalog.test_atul"   # catalog.schema (used only if WRITE_TABLES = True)

# # COMMAND ----------

# # MAGIC %md
# # MAGIC ## 1. Generate the synthetic dataset

# # COMMAND ----------

# rnd = random.Random(SEED)
# TODAY = date.today()

# # ---------------- customers dimension ----------------
# customers = []
# for c in range(1, N_CUSTOMERS + 1):
#     country = rnd.choice(["IN", "IN", "US", "GB"])
#     ctype = rnd.choice(["Retail", "Retail", "SMB", "Enterprise", "Corporate"])
#     customers.append({
#         "customer_id": f"C{c:05d}",
#         "email": f"user{c}@example.com",
#         "country": country,
#         "state": rnd.choice(["MH", "KA", "DL", "TN"]) if country == "IN" else rnd.choice(["CA", "NY", "LDN", None]),
#         "phone": f"+91{rnd.randint(7000000000, 9999999999)}" if country == "IN" else rnd.choice([f"+1{rnd.randint(2000000000, 9999999999)}", None]),
#         "customer_type": ctype,
#         "tax_id": f"TAX{rnd.randint(100000, 999999)}" if ctype in ("Enterprise", "Corporate") else None,
#     })

# customer_schema = T.StructType([
#     T.StructField("customer_id", T.StringType()),
#     T.StructField("email", T.StringType()),
#     T.StructField("country", T.StringType()),
#     T.StructField("state", T.StringType()),
#     T.StructField("phone", T.StringType()),
#     T.StructField("customer_type", T.StringType()),
#     T.StructField("tax_id", T.StringType()),
# ])
# customers_df = spark.createDataFrame(customers, customer_schema)

# # ---------------- orders fact ----------------
# ORDER_STATUSES = ["CREATED", "CONFIRMED", "SHIPPED", "DELIVERED", "CANCELLED", "RETURNED"]
# TXN_STATUSES = ["SUCCESS", "SUCCESS", "SUCCESS", "FAILED", "PENDING", "REFUNDED"]


# def make_order(i: int) -> dict:
#     cust = customers[rnd.randint(0, N_CUSTOMERS - 1)]
#     qty = rnd.randint(1, 10)
#     price = round(rnd.uniform(5, 500), 2)
#     subtotal = round(qty * price, 2)
#     tax = round(subtotal * 0.18, 2)
#     shipping = rnd.choice([0.0, 49.0, 99.0])
#     discount = round(subtotal * rnd.uniform(0, 0.2), 2)
#     total = round(subtotal + tax + shipping - discount, 2)
#     order_status = rnd.choice(ORDER_STATUSES)
#     txn_status = rnd.choice(TXN_STATUSES)
#     ad_spend = round(rnd.uniform(100, 10000), 2)
#     start = TODAY - timedelta(days=rnd.randint(10, 400))
#     case_status = rnd.choice(["OPEN", "CLOSED"])
#     cust_status = rnd.choice(["ACTIVE", "ACTIVE", "INACTIVE"])
#     event_ts = datetime.combine(TODAY, datetime.min.time()) - timedelta(minutes=rnd.randint(0, 6 * 24 * 60))

#     return {
#         "transaction_id": f"TXN{i:06d}",
#         "customer_id": cust["customer_id"],
#         "email": cust["email"],
#         "status": rnd.choice(["active", "pending", "suspended"]),
#         "zipcode": rnd.choice([f"{rnd.randint(10000, 99999)}", f"{rnd.randint(10000, 99999)}-{rnd.randint(1000, 9999)}"]),
#         "country": cust["country"],
#         "state": cust["state"],
#         "phone": cust["phone"],
#         "customer_type": cust["customer_type"],
#         "tax_id": cust["tax_id"],
#         "quantity": qty,
#         "unit_price": price,
#         "subtotal": subtotal,
#         "tax": tax,
#         "shipping_fee": shipping,
#         "discount_amount": discount,
#         "discount_percentage": round(discount / subtotal * 100, 2),
#         "total_amount": total,
#         "transaction_amount": total,
#         "gross_amount": subtotal,
#         "tax_amount": tax,
#         "net_amount": round(subtotal - discount + tax, 2),
#         "currency": rnd.choice(["INR", "INR", "USD", "EUR", "GBP"]),
#         "order_status": order_status,
#         "delivered_at": event_ts + timedelta(days=3) if order_status == "DELIVERED" else None,
#         "cancellation_reason": rnd.choice(["Customer request", "Out of stock"]) if order_status == "CANCELLED" else None,
#         "return_reason": rnd.choice(["Damaged", "Wrong item"]) if order_status == "RETURNED" else None,
#         "transaction_status": txn_status,
#         "refund_amount": total if txn_status == "REFUNDED" else None,
#         "rating": rnd.randint(1, 5),
#         "advertising_spend": ad_spend,
#         "gross_revenue": round(ad_spend * 3.2 + rnd.gauss(0, 1500), 2),   # strongly correlated
#         "start_date": start,
#         "end_date": start + timedelta(days=rnd.randint(0, 30)),
#         "case_status": case_status,
#         "closed_at": event_ts if case_status == "CLOSED" else None,
#         "customer_status": cust_status,
#         "deactivated_at": event_ts if cust_status == "INACTIVE" else None,
#         "event_timestamp": event_ts,
#         "event_date": event_ts.date(),
#         "ingest_age_sec": rnd.randint(0, 20 * 3600),   # within the 24h SLA
#     }


# clean_rows = [make_order(i) for i in range(1, N_ORDERS + 1)]

# # COMMAND ----------

# # MAGIC %md
# # MAGIC ### Inject known defects (dirty copy)
# # MAGIC Each defect targets a specific rule so you can confirm the suite catches it.

# # COMMAND ----------

# import copy

# dirty_rows = copy.deepcopy(clean_rows)


# def r(n):  # 1-based row helper
#     return dirty_rows[n - 1]


# EXPECTED_DEFECTS = []  # (rows, column, rule expected to catch it)


# def defect(rows, desc, fn):
#     for n in rows:
#         fn(r(n))
#     EXPECTED_DEFECTS.append((f"{rows[0]}-{rows[-1]}" if len(rows) > 1 else str(rows[0]), desc))


# defect([1, 2, 3, 4, 5], "total_amount off by +25 → price / total calculation",
#        lambda x: x.update(total_amount=x["total_amount"] + 25))
# defect([6, 7, 8], "Enterprise with NULL tax_id",
#        lambda x: x.update(customer_type="Enterprise", tax_id=None))
# defect([9], "quantity = 0", lambda x: x.update(quantity=0))
# defect([10], "unit_price negative", lambda x: x.update(unit_price=-5.0))
# defect([11], "discount > total", lambda x: x.update(discount_amount=x["total_amount"] + 100))
# defect([12, 13], "SHIPPED with NULL shipping_fee", lambda x: x.update(order_status="SHIPPED", shipping_fee=None))
# defect([14], "DELIVERED with NULL delivered_at", lambda x: x.update(order_status="DELIVERED", delivered_at=None))
# defect([15, 16, 17], "email NULL", lambda x: x.update(email=None))
# defect([18], "email malformed", lambda x: x.update(email="not-an-email"))
# defect([19], "status not in allowed set", lambda x: x.update(status="deleted"))
# defect([20], "zipcode bad pattern", lambda x: x.update(zipcode="ABCDE"))
# defect([21], "duplicate transaction_id (= row 22)", lambda x: x.update(transaction_id="TXN000022"))
# defect([23], "discount_percentage 150", lambda x: x.update(discount_percentage=150.0))
# defect([24], "rating 7", lambda x: x.update(rating=7))
# defect([25], "end_date before start_date", lambda x: x.update(end_date=x["start_date"] - timedelta(days=5)))
# defect([26], "CLOSED with NULL closed_at", lambda x: x.update(case_status="CLOSED", closed_at=None))
# defect([27], "ACTIVE with deactivated_at", lambda x: x.update(customer_status="ACTIVE", deactivated_at=x["event_timestamp"]))
# defect([28], "net_amount off", lambda x: x.update(net_amount=x["net_amount"] + 10))
# defect([29, 30], "ingested 3 days ago (stale)", lambda x: x.update(ingest_age_sec=3 * 24 * 3600))
# defect([31], "IN customer, NULL phone", lambda x: x.update(country="IN", phone=None))
# defect([32], "IN customer, NULL state", lambda x: x.update(country="IN", state=None))
# defect([33], "currency JPY (not allowed)", lambda x: x.update(currency="JPY"))
# defect([34], "currency NULL", lambda x: x.update(currency=None))
# defect([35], "transaction_status UNKNOWN", lambda x: x.update(transaction_status="UNKNOWN"))
# defect([36], "CANCELLED, no reason", lambda x: x.update(order_status="CANCELLED", cancellation_reason=None))
# defect([37], "RETURNED, no reason", lambda x: x.update(order_status="RETURNED", return_reason=None))
# defect([38], "REFUNDED, NULL refund_amount", lambda x: x.update(transaction_status="REFUNDED", refund_amount=None))
# defect([39], "order_status LOST (not allowed)", lambda x: x.update(order_status="LOST"))
# defect([40, 41, 42], "orphan customer_id C99999", lambda x: x.update(customer_id="C99999"))
# defect([43], "transaction_amount negative", lambda x: x.update(transaction_amount=-1.0))
# defect([44], "event_timestamp NULL", lambda x: x.update(event_timestamp=None))

# display(spark.createDataFrame(EXPECTED_DEFECTS, "rows string, defect string"))

# # COMMAND ----------

# order_schema = T.StructType([
#     T.StructField("transaction_id", T.StringType()),
#     T.StructField("customer_id", T.StringType()),
#     T.StructField("email", T.StringType()),
#     T.StructField("status", T.StringType()),
#     T.StructField("zipcode", T.StringType()),
#     T.StructField("country", T.StringType()),
#     T.StructField("state", T.StringType()),
#     T.StructField("phone", T.StringType()),
#     T.StructField("customer_type", T.StringType()),
#     T.StructField("tax_id", T.StringType()),
#     T.StructField("quantity", T.IntegerType()),
#     T.StructField("unit_price", T.DoubleType()),
#     T.StructField("subtotal", T.DoubleType()),
#     T.StructField("tax", T.DoubleType()),
#     T.StructField("shipping_fee", T.DoubleType()),
#     T.StructField("discount_amount", T.DoubleType()),
#     T.StructField("discount_percentage", T.DoubleType()),
#     T.StructField("total_amount", T.DoubleType()),
#     T.StructField("transaction_amount", T.DoubleType()),
#     T.StructField("gross_amount", T.DoubleType()),
#     T.StructField("tax_amount", T.DoubleType()),
#     T.StructField("net_amount", T.DoubleType()),
#     T.StructField("currency", T.StringType()),
#     T.StructField("order_status", T.StringType()),
#     T.StructField("delivered_at", T.TimestampType()),
#     T.StructField("cancellation_reason", T.StringType()),
#     T.StructField("return_reason", T.StringType()),
#     T.StructField("transaction_status", T.StringType()),
#     T.StructField("refund_amount", T.DoubleType()),
#     T.StructField("rating", T.IntegerType()),
#     T.StructField("advertising_spend", T.DoubleType()),
#     T.StructField("gross_revenue", T.DoubleType()),
#     T.StructField("start_date", T.DateType()),
#     T.StructField("end_date", T.DateType()),
#     T.StructField("case_status", T.StringType()),
#     T.StructField("closed_at", T.TimestampType()),
#     T.StructField("customer_status", T.StringType()),
#     T.StructField("deactivated_at", T.TimestampType()),
#     T.StructField("event_timestamp", T.TimestampType()),
#     T.StructField("event_date", T.DateType()),
#     T.StructField("ingest_age_sec", T.LongType()),
# ])


# def to_df(rows) -> DataFrame:
#     """Build the DataFrame and compute ingestion_timestamp relative to Spark's clock."""
#     return (
#         spark.createDataFrame(rows, order_schema)
#         .withColumn(
#             "ingestion_timestamp",
#             (F.unix_timestamp(F.current_timestamp()) - F.col("ingest_age_sec")).cast("timestamp"),
#         )
#         .drop("ingest_age_sec")
#         .cache()
#     )


# clean_df = to_df(clean_rows)
# dirty_df = to_df(dirty_rows)

# print(f"clean_df: {clean_df.count()} rows | dirty_df: {dirty_df.count()} rows | customers_df: {customers_df.count()} rows")
# display(dirty_df.limit(50))

# if WRITE_TABLES:
#     spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET_SCHEMA}")
#     clean_df.write.mode("overwrite").saveAsTable(f"{TARGET_SCHEMA}.orders_clean")
#     dirty_df.write.mode("overwrite").saveAsTable(f"{TARGET_SCHEMA}.orders_dirty")
#     customers_df.write.mode("overwrite").saveAsTable(f"{TARGET_SCHEMA}.customers")


# Read all tables from the target schema
TARGET_SCHEMA = "asper_production_asper_us_dp_dev_catalog.test_atul"

clean_df = spark.table(f"{TARGET_SCHEMA}.orders_clean")
dirty_df = spark.table(f"{TARGET_SCHEMA}.orders_dirty")
customers_df = spark.table(f"{TARGET_SCHEMA}.customers")


# Cache for repeated scans in checks
clean_df.cache()
dirty_df.cache()
customers_df.cache()

# Print counts
print(f"orders_clean: {clean_df.count()} rows")
print(f"orders_dirty: {dirty_df.count()} rows")
print(f"customers: {customers_df.count()} rows")

# Optional: read quarantine if it exists
try:
    quarantine_df = spark.table(f"{TARGET_SCHEMA}.orders_quarantine")
    print(f"orders_quarantine: {quarantine_df.count()} rows")
except:
    print("orders_quarantine table not found (write with WRITE_TABLES=True to create)")

# Preview
display(dirty_df.limit(20))
#display(clean_df.limit(20))
#display(customers_df.limit(20))
display(quarantine_df.limit(20))

orders_clean: 1000 rows
orders_dirty: 1000 rows
customers: 300 rows
orders_quarantine: 21 rows


transaction_id,customer_id,email,status,zipcode,country,state,phone,customer_type,tax_id,quantity,unit_price,subtotal,tax,shipping_fee,discount_amount,discount_percentage,total_amount,transaction_amount,gross_amount,tax_amount,net_amount,currency,order_status,delivered_at,cancellation_reason,return_reason,transaction_status,refund_amount,rating,advertising_spend,gross_revenue,start_date,end_date,case_status,closed_at,customer_status,deactivated_at,event_timestamp,event_date,ingestion_timestamp
TXN000001,C00146,user146@example.com,active,16980-7495,GB,LDN,+17086803230,Enterprise,TAX364742,2,382.74,765.48,137.79,49.0,68.79,8.99,908.48,883.48,765.48,137.79,834.48,INR,CANCELLED,null,Out of stock,null,SUCCESS,null,1,8327.36,26394.05,2026-08-08,2026-09-06,OPEN,null,ACTIVE,null,2026-09-29T00:33:00Z,2026-09-29,2026-10-03T07:02:06Z
TXN000002,C00172,user172@example.com,active,12490,IN,MH,+919371901750,SMB,null,3,73.11,219.33,39.48,49.0,40.24,18.35,292.57,267.57,219.33,39.48,218.57,GBP,DELIVERED,2026-10-05T22:34:00Z,null,null,REFUNDED,267.57,5,8327.99,26903.35,2025-08-31,2025-09-16,CLOSED,2026-10-02T22:34:00Z,ACTIVE,null,2026-10-02T22:34:00Z,2026-10-02,2026-10-02T19:13:57Z
TXN000003,C00057,user57@example.com,suspended,80261,US,LDN,null,Retail,null,5,122.63,613.15,110.37,0.0,5.86,0.96,742.66,717.66,613.15,110.37,717.66,INR,DELIVERED,2026-09-30T07:31:00Z,null,null,REFUNDED,717.66,3,7959.6,23600.18,2026-02-01,2026-02-08,OPEN,null,ACTIVE,null,2026-09-27T07:31:00Z,2026-09-27,2026-10-02T19:28:33Z
TXN000004,C00096,user96@example.com,active,60709-7116,GB,CA,+13955997540,Corporate,TAX853240,2,264.12,528.24,95.08,49.0,7.15,1.35,690.17,665.17,528.24,95.08,616.17,INR,CANCELLED,null,Out of stock,null,PENDING,null,1,5122.54,17283.29,2026-01-07,2026-01-18,OPEN,null,ACTIVE,null,2026-09-27T15:38:00Z,2026-09-27,2026-10-03T01:51:49Z
TXN000005,C00054,user54@example.com,suspended,72722,IN,MH,+919900015820,Corporate,TAX346629,10,368.92,3689.2,664.06,49.0,98.43,2.67,4328.83,4303.83,3689.2,664.06,4254.83,INR,SHIPPED,null,null,null,PENDING,null,4,3451.62,12024.66,2025-10-29,2025-11-23,OPEN,null,INACTIVE,2026-09-27T17:08:00Z,2026-09-27T17:08:00Z,2026-09-27,2026-10-03T03:22:49Z
TXN000006,C00022,user22@example.com,active,95948,GB,CA,+15698408854,Enterprise,null,6,467.02,2802.12,504.38,99.0,223.17,7.96,3182.33,3182.33,2802.12,504.38,3083.33,USD,CANCELLED,null,Out of stock,null,FAILED,null,4,2607.81,8576.79,2025-09-03,2025-09-26,OPEN,null,ACTIVE,null,2026-09-28T22:32:00Z,2026-09-28,2026-10-02T18:37:56Z
TXN000007,C00198,user198@example.com,suspended,81496-5813,GB,CA,null,Enterprise,null,6,486.99,2921.94,525.95,49.0,404.44,13.84,3092.45,3092.45,2921.94,525.95,3043.45,INR,SHIPPED,null,null,null,PENDING,null,1,2740.53,8179.13,2026-08-12,2026-08-26,CLOSED,2026-09-28T02:25:00Z,ACTIVE,null,2026-09-28T02:25:00Z,2026-09-28,2026-10-02T19:07:42Z
TXN000008,C00086,user86@example.com,pending,77251,US,LDN,+19144194790,Enterprise,null,8,179.02,1432.16,257.79,0.0,208.18,14.54,1481.77,1481.77,1432.16,257.79,1481.77,INR,SHIPPED,null,null,null,PENDING,null,5,9981.69,33049.71,2026-05-06,2026-05-27,OPEN,null,ACTIVE,null,2026-09-28T09:05:00Z,2026-09-28,2026-10-02T18:41:21Z
TXN000009,C00018,user18@example.com,pending,85225-3255,IN,TN,+918149938334,SMB,null,0,38.28,114.84,20.67,99.0,8.41,7.32,226.1,226.1,114.84,20.67,127.1,EUR,DELIVERED,2026-10-01T18:25:00Z,null,null,PENDING,null,3,420.61,675.24,2026-07-07,2026-07-10,CLOSED,2026-09-28T18:25:00Z,ACTIVE,null,2026-09-28T18:25:00Z,2026-09-28,2026-10-03T09:41:57Z
TXN000010,C00096,user96@example.com,active,27790-2213,GB,CA,+13955997540,Corporate,TAX853240,8,-5.0,2090.24,376.24,99.0,49.23,2.36,2516.25,2516.25,2090.24,376.24,2417.25,INR,SHIPPED,null,null,null,REFUNDED,2516.25,4,4518.1,13540.1,2025-11-14,2025-11-16,CLOSED,2026-09-27T09:53:00Z,INACTIVE,2026-09-27T09:53:00Z,2026-09-27T09:53:00Z,2026-09-27,2026-10-02T23:00:24Z


transaction_id,customer_id,email,status,zipcode,country,state,phone,customer_type,tax_id,quantity,unit_price,subtotal,tax,shipping_fee,discount_amount,discount_percentage,total_amount,transaction_amount,gross_amount,tax_amount,net_amount,currency,order_status,delivered_at,cancellation_reason,return_reason,transaction_status,refund_amount,rating,advertising_spend,gross_revenue,start_date,end_date,case_status,closed_at,customer_status,deactivated_at,event_timestamp,event_date,ingestion_timestamp,failed_rules,quarantined_at
TXN000001,C00146,user146@example.com,active,16980-7495,GB,LDN,+17086803230,Enterprise,TAX364742,2,382.74,765.48,137.79,49.0,68.79,8.99,908.48,883.48,765.48,137.79,834.48,INR,CANCELLED,null,Out of stock,null,SUCCESS,null,1,8327.36,26394.05,2026-08-08,2026-09-06,OPEN,null,ACTIVE,null,2026-09-29T00:33:00Z,2026-09-29,2026-10-03T07:02:06Z,List(price_calculation),2026-10-03T11:05:36.34Z
TXN000002,C00172,user172@example.com,active,12490,IN,MH,+919371901750,SMB,null,3,73.11,219.33,39.48,49.0,40.24,18.35,292.57,267.57,219.33,39.48,218.57,GBP,DELIVERED,2026-10-05T22:34:00Z,null,null,REFUNDED,267.57,5,8327.99,26903.35,2025-08-31,2025-09-16,CLOSED,2026-10-02T22:34:00Z,ACTIVE,null,2026-10-02T22:34:00Z,2026-10-02,2026-10-02T19:13:57Z,List(price_calculation),2026-10-03T11:05:36.34Z
TXN000003,C00057,user57@example.com,suspended,80261,US,LDN,null,Retail,null,5,122.63,613.15,110.37,0.0,5.86,0.96,742.66,717.66,613.15,110.37,717.66,INR,DELIVERED,2026-09-30T07:31:00Z,null,null,REFUNDED,717.66,3,7959.6,23600.18,2026-02-01,2026-02-08,OPEN,null,ACTIVE,null,2026-09-27T07:31:00Z,2026-09-27,2026-10-02T19:28:33Z,List(price_calculation),2026-10-03T11:05:36.34Z
TXN000004,C00096,user96@example.com,active,60709-7116,GB,CA,+13955997540,Corporate,TAX853240,2,264.12,528.24,95.08,49.0,7.15,1.35,690.17,665.17,528.24,95.08,616.17,INR,CANCELLED,null,Out of stock,null,PENDING,null,1,5122.54,17283.29,2026-01-07,2026-01-18,OPEN,null,ACTIVE,null,2026-09-27T15:38:00Z,2026-09-27,2026-10-03T01:51:49Z,List(price_calculation),2026-10-03T11:05:36.34Z
TXN000005,C00054,user54@example.com,suspended,72722,IN,MH,+919900015820,Corporate,TAX346629,10,368.92,3689.2,664.06,49.0,98.43,2.67,4328.83,4303.83,3689.2,664.06,4254.83,INR,SHIPPED,null,null,null,PENDING,null,4,3451.62,12024.66,2025-10-29,2025-11-23,OPEN,null,INACTIVE,2026-09-27T17:08:00Z,2026-09-27T17:08:00Z,2026-09-27,2026-10-03T03:22:49Z,List(price_calculation),2026-10-03T11:05:36.34Z
TXN000006,C00022,user22@example.com,active,95948,GB,CA,+15698408854,Enterprise,null,6,467.02,2802.12,504.38,99.0,223.17,7.96,3182.33,3182.33,2802.12,504.38,3083.33,USD,CANCELLED,null,Out of stock,null,FAILED,null,4,2607.81,8576.79,2025-09-03,2025-09-26,OPEN,null,ACTIVE,null,2026-09-28T22:32:00Z,2026-09-28,2026-10-02T18:37:56Z,List(enterprise_tax_id),2026-10-03T11:05:36.34Z
TXN000007,C00198,user198@example.com,suspended,81496-5813,GB,CA,null,Enterprise,null,6,486.99,2921.94,525.95,49.0,404.44,13.84,3092.45,3092.45,2921.94,525.95,3043.45,INR,SHIPPED,null,null,null,PENDING,null,1,2740.53,8179.13,2026-08-12,2026-08-26,CLOSED,2026-09-28T02:25:00Z,ACTIVE,null,2026-09-28T02:25:00Z,2026-09-28,2026-10-02T19:07:42Z,List(enterprise_tax_id),2026-10-03T11:05:36.34Z
TXN000008,C00086,user86@example.com,pending,77251,US,LDN,+19144194790,Enterprise,null,8,179.02,1432.16,257.79,0.0,208.18,14.54,1481.77,1481.77,1432.16,257.79,1481.77,INR,SHIPPED,null,null,null,PENDING,null,5,9981.69,33049.71,2026-05-06,2026-05-27,OPEN,null,ACTIVE,null,2026-09-28T09:05:00Z,2026-09-28,2026-10-02T18:41:21Z,List(enterprise_tax_id),2026-10-03T11:05:36.34Z
TXN000009,C00018,user18@example.com,pending,85225-3255,IN,TN,+918149938334,SMB,null,0,38.28,114.84,20.67,99.0,8.41,7.32,226.1,226.1,114.84,20.67,127.1,EUR,DELIVERED,2026-10-01T18:25:00Z,null,null,PENDING,null,3,420.61,675.24,2026-07-07,2026-07-10,CLOSED,2026-09-28T18:25:00Z,ACTIVE,null,2026-09-28T18:25:00Z,2026-09-28,2026-10-03T09:41:57Z,"List(price_calculation, quantity_positive)",2026-10-03T11:05:36.34Z
TXN000010,C

### More advanced check Satisfies,Hascorrelation

In [0]:
ALL_RESULTS = []   # collected for the final summary


def results_df(verification_result) -> DataFrame:
    return VerificationResult.checkResultsAsDataFrame(spark, verification_result)


def run_check_suite(df: DataFrame, check: Check, suite_name: str, dataset: str = "dirty"):
    """Run a PyDeequ verification suite, display results, and store them for the summary."""
    result = VerificationSuite(spark).onData(df).addCheck(check).run()
    print(f"\n{'=' * 80}\nRESULT: {suite_name} [{dataset}]  →  {getattr(result, 'status', '')}\n{'=' * 80}")
    rdf = results_df(result)
    display(rdf.select("constraint", "constraint_status", "constraint_message"))
    ALL_RESULTS.append(rdf.withColumn("suite", F.lit(suite_name)).withColumn("dataset", F.lit(dataset)))
    return result

In [0]:
advanced_check = (
    Check(spark, CheckLevel.Error, "Advanced Schema and Business Validation")
    .satisfies("ABS(total_amount - (quantity * unit_price + tax + shipping_fee - discount_amount)) < 0.01",
               "Price calculation mismatch")
    .hasCorrelation("advertising_spend", "gross_revenue", lambda corr: corr > 0.7)
    .satisfies("CASE WHEN customer_type = 'Enterprise' THEN tax_id IS NOT NULL ELSE true END",
               "Missing tax_id for corporate clients")
    .satisfies("quantity > 0", "Quantity must be greater than zero")
    .satisfies("unit_price >= 0 AND tax >= 0", "Unit price or tax is negative")
    .satisfies("total_amount >= 0", "Total amount cannot be negative")
    .satisfies("discount_amount <= total_amount", "Discount cannot exceed total amount")
    .satisfies("CASE WHEN order_status = 'SHIPPED' THEN shipping_fee IS NOT NULL ELSE true END",
               "Shipping fee missing for shipped order")
    .satisfies("CASE WHEN order_status = 'DELIVERED' THEN delivered_at IS NOT NULL ELSE true END",
               "Delivered order has no delivered_at timestamp")
)

run_check_suite(clean_df, advanced_check, "Advanced Business Validation", "clean")
run_check_suite(dirty_df, advanced_check, "Advanced Business Validation", "dirty")


RESULT: Advanced Business Validation [clean]  →  Success


/databricks/spark/python/pyspark/sql/dataframe.py:160: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


constraint,constraint_status,constraint_message
"ComplianceConstraint(Compliance(Price calculation mismatch,ABS(total_amount - (quantity * unit_price + tax + shipping_fee - discount_amount)) < 0.01,None,List(),None))",Success,
"CorrelationConstraint(Correlation(advertising_spend,gross_revenue,None))",Success,
"ComplianceConstraint(Compliance(Missing tax_id for corporate clients,CASE WHEN customer_type = 'Enterprise' THEN tax_id IS NOT NULL ELSE true END,None,List(),None))",Success,
"ComplianceConstraint(Compliance(Quantity must be greater than zero,quantity > 0,None,List(),None))",Success,
"ComplianceConstraint(Compliance(Unit price or tax is negative,unit_price >= 0 AND tax >= 0,None,List(),None))",Success,
"ComplianceConstraint(Compliance(Total amount cannot be negative,total_amount >= 0,None,List(),None))",Success,
"ComplianceConstraint(Compliance(Discount cannot exceed total amount,discount_amount <= total_amount,None,List(),None))",Success,
"ComplianceConstraint(Compliance(Shipping fee missing for shipped order,CASE WHEN order_status = 'SHIPPED' THEN shipping_fee IS NOT NULL ELSE true END,None,List(),None))",Success,
"ComplianceConstraint(Compliance(Delivered order has no delivered_at timestamp,CASE WHEN order_status = 'DELIVERED' THEN delivered_at IS NOT NULL ELSE true END,None,List(),None))",Success,



RESULT: Advanced Business Validation [dirty]  →  Error


constraint,constraint_status,constraint_message
"ComplianceConstraint(Compliance(Price calculation mismatch,ABS(total_amount - (quantity * unit_price + tax + shipping_fee - discount_amount)) < 0.01,None,List(),None))",Failure,Value: 0.99 does not meet the constraint requirement!
"CorrelationConstraint(Correlation(advertising_spend,gross_revenue,None))",Success,
"ComplianceConstraint(Compliance(Missing tax_id for corporate clients,CASE WHEN customer_type = 'Enterprise' THEN tax_id IS NOT NULL ELSE true END,None,List(),None))",Failure,Value: 0.997 does not meet the constraint requirement!
"ComplianceConstraint(Compliance(Quantity must be greater than zero,quantity > 0,None,List(),None))",Failure,Value: 0.999 does not meet the constraint requirement!
"ComplianceConstraint(Compliance(Unit price or tax is negative,unit_price >= 0 AND tax >= 0,None,List(),None))",Failure,Value: 0.999 does not meet the constraint requirement!
"ComplianceConstraint(Compliance(Total amount cannot be negative,total_amount >= 0,None,List(),None))",Success,
"ComplianceConstraint(Compliance(Discount cannot exceed total amount,discount_amount <= total_amount,None,List(),None))",Failure,Value: 0.999 does not meet the constraint requirement!
"ComplianceConstraint(Compliance(Shipping fee missing for shipped order,CASE WHEN order_status = 'SHIPPED' THEN shipping_fee IS NOT NULL ELSE true END,None,List(),None))",Failure,Value: 0.998 does not meet the constraint requirement!
"ComplianceConstraint(Compliance(Delivered order has no delivered_at timestamp,CASE WHEN order_status = 'DELIVERED' THEN delivered_at IS NOT NULL ELSE true END,None,List(),None))",Failure,Value: 0.999 does not meet the constraint requirement!


In [0]:
from pyspark.sql import SparkSession
import pydeequ

spark = (SparkSession.builder
    .appName("deequ")
    .config("spark.jars.packages", pydeequ.deequ_maven_coord)
    .config("spark.jars.excludes", pydeequ.f2j_maven_coord)
    .getOrCreate()
)

### Numeric Range validation

In [0]:
numeric_range_check = (
    Check(spark, CheckLevel.Error, "Numeric Range Validation")
    .isNonNegative("quantity")
    .isNonNegative("unit_price")
    .isNonNegative("tax")
    .isNonNegative("total_amount")
    .satisfies("discount_percentage >= 0 AND discount_percentage <= 100", "Discount percentage is outside 0-100")
    .satisfies("rating >= 1 AND rating <= 5", "Rating must be between 1 and 5")
)

run_check_suite(clean_df, numeric_range_check, "Numeric Range Validation", "clean")
run_check_suite(dirty_df, numeric_range_check, "Numeric Range Validation", "dirty")


RESULT: Numeric Range Validation [clean]  →  Success


/databricks/spark/python/pyspark/sql/dataframe.py:160: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


constraint,constraint_status,constraint_message
"ComplianceConstraint(Compliance(quantity is non-negative,COALESCE(CAST(quantity AS DECIMAL(20,10)), 0.0) >= 0,None,List(quantity),None))",Success,
"ComplianceConstraint(Compliance(unit_price is non-negative,COALESCE(CAST(unit_price AS DECIMAL(20,10)), 0.0) >= 0,None,List(unit_price),None))",Success,
"ComplianceConstraint(Compliance(tax is non-negative,COALESCE(CAST(tax AS DECIMAL(20,10)), 0.0) >= 0,None,List(tax),None))",Success,
"ComplianceConstraint(Compliance(total_amount is non-negative,COALESCE(CAST(total_amount AS DECIMAL(20,10)), 0.0) >= 0,None,List(total_amount),None))",Success,
"ComplianceConstraint(Compliance(Discount percentage is outside 0-100,discount_percentage >= 0 AND discount_percentage <= 100,None,List(),None))",Success,
"ComplianceConstraint(Compliance(Rating must be between 1 and 5,rating >= 1 AND rating <= 5,None,List(),None))",Success,



RESULT: Numeric Range Validation [dirty]  →  Error


constraint,constraint_status,constraint_message
"ComplianceConstraint(Compliance(quantity is non-negative,COALESCE(CAST(quantity AS DECIMAL(20,10)), 0.0) >= 0,None,List(quantity),None))",Success,
"ComplianceConstraint(Compliance(unit_price is non-negative,COALESCE(CAST(unit_price AS DECIMAL(20,10)), 0.0) >= 0,None,List(unit_price),None))",Failure,Value: 0.999 does not meet the constraint requirement!
"ComplianceConstraint(Compliance(tax is non-negative,COALESCE(CAST(tax AS DECIMAL(20,10)), 0.0) >= 0,None,List(tax),None))",Success,
"ComplianceConstraint(Compliance(total_amount is non-negative,COALESCE(CAST(total_amount AS DECIMAL(20,10)), 0.0) >= 0,None,List(total_amount),None))",Success,
"ComplianceConstraint(Compliance(Discount percentage is outside 0-100,discount_percentage >= 0 AND discount_percentage <= 100,None,List(),None))",Failure,Value: 0.999 does not meet the constraint requirement!
"ComplianceConstraint(Compliance(Rating must be between 1 and 5,rating >= 1 AND rating <= 5,None,List(),None))",Failure,Value: 0.999 does not meet the constraint requirement!


###  MAGIC 13. Cross-column consistency


In [0]:
# COMMAND ----------

# MAGIC %md
# MAGIC ## 13. Cross-column consistency

# COMMAND ----------

cross_column_check = (
    Check(spark, CheckLevel.Error, "Cross Column Consistency")
    .satisfies("end_date IS NULL OR start_date IS NULL OR end_date >= start_date", "End date occurs before start date")
    .satisfies("CASE WHEN case_status = 'CLOSED' THEN closed_at IS NOT NULL ELSE true END",
               "Closed case has no closed_at timestamp")
    .satisfies("CASE WHEN customer_status = 'ACTIVE' THEN deactivated_at IS NULL ELSE true END",
               "Active customer has deactivation date")
    .satisfies("ABS(net_amount - (gross_amount - discount_amount + tax_amount)) < 0.01",
               "Net amount calculation mismatch")
)

run_check_suite(clean_df, cross_column_check, "Cross Column Consistency", "clean")
run_check_suite(dirty_df, cross_column_check, "Cross Column Consistency", "dirty")


RESULT: Cross Column Consistency [clean]  →  Success


/databricks/spark/python/pyspark/sql/dataframe.py:160: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


constraint,constraint_status,constraint_message
"ComplianceConstraint(Compliance(End date occurs before start date,end_date IS NULL OR start_date IS NULL OR end_date >= start_date,None,List(),None))",Success,
"ComplianceConstraint(Compliance(Closed case has no closed_at timestamp,CASE WHEN case_status = 'CLOSED' THEN closed_at IS NOT NULL ELSE true END,None,List(),None))",Success,
"ComplianceConstraint(Compliance(Active customer has deactivation date,CASE WHEN customer_status = 'ACTIVE' THEN deactivated_at IS NULL ELSE true END,None,List(),None))",Success,
"ComplianceConstraint(Compliance(Net amount calculation mismatch,ABS(net_amount - (gross_amount - discount_amount + tax_amount)) < 0.01,None,List(),None))",Success,



RESULT: Cross Column Consistency [dirty]  →  Error


constraint,constraint_status,constraint_message
"ComplianceConstraint(Compliance(End date occurs before start date,end_date IS NULL OR start_date IS NULL OR end_date >= start_date,None,List(),None))",Failure,Value: 0.999 does not meet the constraint requirement!
"ComplianceConstraint(Compliance(Closed case has no closed_at timestamp,CASE WHEN case_status = 'CLOSED' THEN closed_at IS NOT NULL ELSE true END,None,List(),None))",Failure,Value: 0.999 does not meet the constraint requirement!
"ComplianceConstraint(Compliance(Active customer has deactivation date,CASE WHEN customer_status = 'ACTIVE' THEN deactivated_at IS NULL ELSE true END,None,List(),None))",Failure,Value: 0.999 does not meet the constraint requirement!
"ComplianceConstraint(Compliance(Net amount calculation mismatch,ABS(net_amount - (gross_amount - discount_amount + tax_amount)) < 0.01,None,List(),None))",Failure,Value: 0.998 does not meet the constraint requirement!


###  MAGIC 15. Conditional (null-combination) rules

In [0]:
# COMMAND ----------

# MAGIC %md


# COMMAND ----------

conditional_check = (
    Check(spark, CheckLevel.Error, "Conditional Data Quality")
    .satisfies("CASE WHEN country = 'IN' THEN phone IS NOT NULL ELSE true END", "Phone missing for Indian customers")
    .satisfies("CASE WHEN country = 'IN' THEN state IS NOT NULL ELSE true END", "State missing for Indian customers")
    .satisfies("CASE WHEN customer_type IN ('Enterprise', 'Corporate') THEN tax_id IS NOT NULL ELSE true END",
               "Tax ID missing for corporate customer")
)

run_check_suite(clean_df, conditional_check, "Conditional Data Quality", "clean")
run_check_suite(dirty_df, conditional_check, "Conditional Data Quality", "dirty")




RESULT: Conditional Data Quality [clean]  →  Success


/databricks/spark/python/pyspark/sql/dataframe.py:160: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


constraint,constraint_status,constraint_message
"ComplianceConstraint(Compliance(Phone missing for Indian customers,CASE WHEN country = 'IN' THEN phone IS NOT NULL ELSE true END,None,List(),None))",Success,
"ComplianceConstraint(Compliance(State missing for Indian customers,CASE WHEN country = 'IN' THEN state IS NOT NULL ELSE true END,None,List(),None))",Success,
"ComplianceConstraint(Compliance(Tax ID missing for corporate customer,CASE WHEN customer_type IN ('Enterprise', 'Corporate') THEN tax_id IS NOT NULL ELSE true END,None,List(),None))",Success,



RESULT: Conditional Data Quality [dirty]  →  Error


constraint,constraint_status,constraint_message
"ComplianceConstraint(Compliance(Phone missing for Indian customers,CASE WHEN country = 'IN' THEN phone IS NOT NULL ELSE true END,None,List(),None))",Failure,Value: 0.999 does not meet the constraint requirement!
"ComplianceConstraint(Compliance(State missing for Indian customers,CASE WHEN country = 'IN' THEN state IS NOT NULL ELSE true END,None,List(),None))",Failure,Value: 0.999 does not meet the constraint requirement!
"ComplianceConstraint(Compliance(Tax ID missing for corporate customer,CASE WHEN customer_type IN ('Enterprise', 'Corporate') THEN tax_id IS NOT NULL ELSE true END,None,List(),None))",Failure,Value: 0.997 does not meet the constraint requirement!


###  MAGIC  16. Status / business-state validation

In [0]:
# COMMAND ----------

# MAGIC %md


# COMMAND ----------

status_check = (
    Check(spark, CheckLevel.Error, "Status Validation")
    .satisfies("CASE WHEN order_status = 'CANCELLED' THEN cancellation_reason IS NOT NULL ELSE true END",
               "Cancelled order has no cancellation reason")
    .satisfies("CASE WHEN order_status = 'RETURNED' THEN return_reason IS NOT NULL ELSE true END",
               "Returned order has no return reason")
)

run_check_suite(clean_df, status_check, "Status Validation", "clean")
run_check_suite(dirty_df, status_check, "Status Validation", "dirty")


RESULT: Status Validation [clean]  →  Success


/databricks/spark/python/pyspark/sql/dataframe.py:160: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


constraint,constraint_status,constraint_message
"ComplianceConstraint(Compliance(Cancelled order has no cancellation reason,CASE WHEN order_status = 'CANCELLED' THEN cancellation_reason IS NOT NULL ELSE true END,None,List(),None))",Success,
"ComplianceConstraint(Compliance(Returned order has no return reason,CASE WHEN order_status = 'RETURNED' THEN return_reason IS NOT NULL ELSE true END,None,List(),None))",Success,



RESULT: Status Validation [dirty]  →  Error


constraint,constraint_status,constraint_message
"ComplianceConstraint(Compliance(Cancelled order has no cancellation reason,CASE WHEN order_status = 'CANCELLED' THEN cancellation_reason IS NOT NULL ELSE true END,None,List(),None))",Failure,Value: 0.999 does not meet the constraint requirement!
"ComplianceConstraint(Compliance(Returned order has no return reason,CASE WHEN order_status = 'RETURNED' THEN return_reason IS NOT NULL ELSE true END,None,List(),None))",Failure,Value: 0.999 does not meet the constraint requirement!


In [0]:
def get_check_results(result):
    """Compatible with PyDeequ 1.3.0"""
    try:
        # Try newer syntax first
        return result.checkResultsAsDataFrame()
    except TypeError:
        # Fall back to older syntax
        return VerificationResult.checkResultsAsDataFrame(spark, result)

# ============================================================================
# TEST CASE 1: COMPLETENESS CHECKS
# ============================================================================
def test_completeness():
    print("\n" + "="*60)
    print("TEST 1: COMPLETENESS CHECKS")
    print("="*60)
    
    df = spark.createDataFrame([
        (1, "Alice", "alice@example.com", "2024-01-01"),
        (2, "Bob", None, "2024-01-02"),
        (3, None, "charlie@example.com", None),
        (4, "Diana", "diana@example.com", "2024-01-04"),
        (5, "Eve", "eve@example.com", "2024-01-05"),
    ], ["id", "name", "email", "date"])
    
    result = (VerificationSuite(spark)
        .onData(df)
        .addCheck(
            Check(spark, CheckLevel.Error, "Completeness")
            .hasCompleteness("id", lambda x: x > 0.95)
            .hasCompleteness("name", lambda x: x > 0.8)
            .hasCompleteness("email", lambda x: x > 0.8)
            .hasCompleteness("date", lambda x: x > 0.6)
        )
        .run()
    )
    # FIX: Use helper function
    print(get_check_results(result).show(truncate=False))


# ============================================================================
# TEST CASE 2: UNIQUENESS & DUPLICATE DETECTION
# ============================================================================
def test_uniqueness():
    print("\n" + "="*60)
    print("TEST 2: UNIQUENESS & DUPLICATE DETECTION")
    print("="*60)
    
    df = spark.createDataFrame([
        (1, "alice@example.com", "US"),
        (2, "bob@example.com", "UK"),
        (3, "alice@example.com", "US"),
        (4, "charlie@example.com", "CA"),
        (5, "charlie@example.com", "CA"),
    ], ["id", "email", "country"])
    
    result = (VerificationSuite(spark)
        .onData(df)
        .addCheck(
            Check(spark, CheckLevel.Error, "Uniqueness")
            .isUnique("id")
            .isUnique("email")
        )
        .run()
    )
    print(get_check_results(result).show(truncate=False))


# ============================================================================
# TEST CASE 3: RANGE & PATTERN VALIDATION
# ============================================================================
def test_patterns():
    print("\n" + "="*60)
    print("TEST 3: RANGE & PATTERN VALIDATION")
    print("="*60)
    
    df = spark.createDataFrame([
        (1, "alice@example.com", 25, "2024-01-01"),
        (2, "bob_invalid", 150, "invalid-date"),
        (3, "charlie@example.com", -5, "2024-01-03"),
        (4, "diana@example.com", 35, "2024-01-04"),
    ], ["id", "email", "age", "signup_date"])
    
    result = (VerificationSuite(spark)
        .onData(df)
        .addCheck(
            Check(spark, CheckLevel.Error, "Patterns & Ranges")
            .containsEmail("email", lambda x: x > 0.5)
            
            .hasPattern("signup_date", r"^\d{4}-\d{2}-\d{2}$", lambda x: x > 0.8)
        )
        .run()
    )
    print(get_check_results(result).show(truncate=False))





# ============================================================================
# TEST CASE 5: DISTRIBUTIONS
# ============================================================================
def test_distributions():
    print("\n" + "="*60)
    print("TEST 5: VALUE DISTRIBUTION CHECKS")
    print("="*60)
    
    df = spark.createDataFrame([
        (1, "active", "US"),
        (2, "active", "UK"),
        (3, "inactive", "US"),
        (4, "active", "CA"),
        (5, "pending", "US"),
        (6, "active", "US"),
    ], ["id", "status", "country"])
    
    result = (VerificationSuite(spark)
        .onData(df)
        .addCheck(
            Check(spark, CheckLevel.Warning, "Distributions")
            .isContainedIn("status", ["active", "inactive", "pending"], lambda x: x > 0.9)
            .isContainedIn("country", ["US", "UK", "CA", "IN"], lambda x: x > 0.9)
        )
        .run()
    )
    print(get_check_results(result).show(truncate=False))


# ============================================================================
# TEST CASE 6: MULTIPLE SEVERITY LEVELS
# ============================================================================
def test_severity_levels():
    print("\n" + "="*60)
    print("TEST 6: MULTIPLE SEVERITY LEVELS")
    print("="*60)
    
    df = spark.createDataFrame([
        (1, "alice@example.com", 25),
        (2, "bob_invalid", 30),
        (3, "charlie@example.com", None),
    ], ["id", "email", "age"])
    
    result = (VerificationSuite(spark)
        .onData(df)
        .addCheck(
            Check(spark, CheckLevel.Error, "Critical Rules")
            .hasCompleteness("id", lambda x: x >= 1.0)
            .isUnique("id")
        )
        .addCheck(
            Check(spark, CheckLevel.Warning, "Data Quality")
            .hasCompleteness("email", lambda x: x > 0.8)
            .hasCompleteness("age", lambda x: x > 0.8)
            .containsEmail("email", lambda x: x > 0.5)
        )
        .run()
    )
    print(get_check_results(result).show(truncate=False))


# ============================================================================
# TEST CASE 7: SIZE CHECKS
# ============================================================================
def test_size_checks():
    print("\n" + "="*60)
    print("TEST 7: ROW COUNT CHECKS")
    print("="*60)
    
    df = spark.createDataFrame([(i, f"user_{i}") for i in range(100)], ["id", "name"])
    
    result = (VerificationSuite(spark)
        .onData(df)
        .addCheck(
            Check(spark, CheckLevel.Error, "Size Checks")
            .hasSize(lambda x: x > 50)
            .hasSize(lambda x: x < 1000)
        )
        .run()
    )
    print(get_check_results(result).show(truncate=False))


# ============================================================================
# RUN ALL TESTS
# ============================================================================
if __name__ == "__main__":
    test_completeness()
    test_uniqueness()
    test_patterns()
    
    test_distributions()
    test_severity_levels()
    test_size_checks()
    
    print("\n" + "="*60)
    print("✅ ALL TESTS COMPLETED")
    print("="*60)


TEST 1: COMPLETENESS CHECKS


/databricks/spark/python/pyspark/sql/dataframe.py:160: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


+------------+-----------+------------+-----------------------------------------------------+-----------------+----------------------------------------------------+
|check       |check_level|check_status|constraint                                           |constraint_status|constraint_message                                  |
+------------+-----------+------------+-----------------------------------------------------+-----------------+----------------------------------------------------+
|Completeness|Error      |Error       |CompletenessConstraint(Completeness(id,None,None))   |Success          |                                                    |
|Completeness|Error      |Error       |CompletenessConstraint(Completeness(name,None,None)) |Failure          |Value: 0.8 does not meet the constraint requirement!|
|Completeness|Error      |Error       |CompletenessConstraint(Completeness(email,None,None))|Failure          |Value: 0.8 does not meet the constraint requirement!|
|Completen